In [ ]:
import sys; sys.path.append('../../'); sys.path.append('../../periodic_patches/'); sys.path.append('../experiments/'); sys.path.append('../../gmsh')
import inflation, mesh, numpy as np, importlib, pickle, gzip, boundaries, utils, scipy, benchmark


In [ ]:
isheet = pickle.load(gzip.open('igloo_sim.pkl.gz', 'r'))

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output


from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
# Fit plane to bdry to get gravity direction +/-
fixedVars = boundaries.getBoundaryVars(isheet)
bdryVertices = []
for fv in fixedVars:
    a = isheet.vtxForVar(fv)
    assert a.sheet == 3
    if a.vi not in bdryVertices: bdryVertices.append(a.vi)
bdryVertices = isheet.mesh().vertices()[bdryVertices]
u,s,v = np.linalg.svd(bdryVertices.transpose()@ bdryVertices)
gravity_dir = v[s.argmin()]
gravity_dir

In [ ]:
isheet.pressure = 1e-2
isheet.rho = 1e-6
isheet.gravity = -9.80635 * gravity_dir

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars,  opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()